In [1]:
import pandas as pd
from transformers import pipeline , AutoTokenizer , AutoModelForSequenceClassification
import torch

In [2]:
def score_text(text, hyps, batch_size=8):
    scores = []
    for i in range(0, len(hyps), batch_size):
        batch = hyps[i:i+batch_size]
        inputs = tok([text] * len(batch), batch,
                     return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits          # 3 numbers per pair
        probs = torch.softmax(logits.float(), dim=-1) 
        scores += probs[:, ent_idx].tolist()         # keep only entailment
    return scores                                    

In [4]:
name = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, dtype=torch.float32).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
ent_idx = model.config.label2id["entailment"]
print(model.config.label2id) 
print(model.config.dtype) 

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

{'contradiction': 2, 'entailment': 0, 'neutral': 1}
torch.float32


In [7]:
# this to test is problem in  the model or our predict function 
premise = "America is SUPPOSED to open her borders to genuine refugees HELLO this is America! We ALL were refugees of one sort or another."
controls = [
    "This text is about a football match.",
    "This text contains a cooking recipe.",
]
concrete = [
    "The text describes refugees as a flood or a wave.",
    "The text describes refugees as an invasion or an attack.",
    "The text describes refugees as vermin or a disease.",
    "The text uses the word 'door', 'wall' or 'house' to talk about borders.",
    "The text says that the government should do something.",
    "The text says that some people disagree about refugees.",
]
print(f"premise :{premise}")
for h in controls + concrete:
    print(round(score_text(premise, [h])[0], 3), h)

premise :America is SUPPOSED to open her borders to genuine refugees HELLO this is America! We ALL were refugees of one sort or another.
0.0 This text is about a football match.
0.0 This text contains a cooking recipe.
0.0 The text describes refugees as a flood or a wave.
0.0 The text describes refugees as an invasion or an attack.
0.0 The text describes refugees as vermin or a disease.
0.529 The text uses the word 'door', 'wall' or 'house' to talk about borders.
0.989 The text says that the government should do something.
0.549 The text says that some people disagree about refugees.


In [8]:
# chage name of file and sheet 's name  if they are diffrent in ur pc
df_hypothesis = pd.read_excel("Human_pilot_llm_pipeline.xlsx",sheet_name="Hypothesis Bank")
df_tweets = pd.read_excel("refugee_tweets_deduplicated.xlsx")

In [18]:
# some vectors used as  columns names 
hypothesis_list = df_hypothesis["hypothesis"].to_list()
ids  = df_hypothesis["id"].tolist()
frames = df_hypothesis["frame / domain"].tolist()

In [22]:
# seperated per new york times article
ny_ranges=df_tweets[df_tweets["Handle"] == "@nytimes"].index
length_ny_ranges = len(ny_ranges)
tweets_separated_per_article = [
    df_tweets.iloc[ny_ranges[i]:ny_ranges[i + 1]]
    if i + 1 < len(ny_ranges)
    else df_tweets.iloc[ny_ranges[i]:]
    for i in range(0, len(ny_ranges), 1)
]

,Handle,Date,Set,Text
56,@nytimes,"Mar 19, 2022",Ukrainian Refugee Crisis,ARTICLE: The first Ukrainian refugees are tric...
57,@1MansOpinionz,"Mar 19, 2022",Ukrainian Refugee Crisis,Because they are white?
58,@WilliamsNietzs3,"Mar 19, 2022",Ukrainian Refugee Crisis,While I strongly disagree with the Biden admin...
59,@AlexoV2,"Mar 19, 2022",Ukrainian Refugee Crisis,Biden was escalating this war so it is his res...
60,@DeborahElaineW1,"Mar 19, 2022",Ukrainian Refugee Crisis,United States is the destination of centuries ...
61,@Arya1234Stark,"Mar 19, 2022",Ukrainian Refugee Crisis,Europe doesn't have the means to house ten mil...
62,@LordStarjun,"Mar 19, 2022",Ukrainian Refugee Crisis,Mexico? Central America? South America? Haiti?...
63,@stepoya,"Mar 19, 2022",Ukrainian Refugee Crisis,[link to blog post: oyagah.blogspot.com]
64,@SofianeGharbi11,"Mar 19, 2022",Ukrainian Refugee Crisis,Stop war in Yemen 🇾🇪
65,@EPenalvert,"Mar 19, 2022",Ukrainian Refugee Crisis,THE NYT WITH THEIR FAKE NEWS GAVE US BIDEN. AN...


In [26]:
#here we are testing only the frist range that follow article from 1->11
ny_first_range = tweets_separated_per_article[0]['Text'].to_list()
rows = [score_text(t, hypothesis_list) for t in test]

In [27]:
ny_first_range

['ARTICLE: The U.S. agreed to accept up to 2,000 Syrian refugees, prompting pushback by G.O.P. lawmakers (http://nyti.ms/1JQTDoe)',
 'America is SUPPOSED to open her borders to genuine refugees HELLO this is America! We ALL were refugees of one sort or another.',
 'If we do not provide humanitarian relief, we are supporting their potential genocide. That makes us no better than the terrorists.',
 "America always except the downtrodden's from their countries that their government don't take care of them.",
 'these are real refugees unlike illegal Mexicans coming over just because they feel like it.',
 "2000? That's a piss in the wind. #selfishamericans",
 "can't we stop being the world's dumpster?",
 'I hate the GOP - HATE THEM - have hated them ever since the Sarah Palin nightmare and they have only gone downhill since then.',
 'Forget it! No good deed goes unpunished!',
 'European countries take in more refugees by country and make the US look extremely bad... the GOP, destroying the 

In [25]:
scores_raw = pd.DataFrame(rows, columns=frames).round(3)

,Responsibility,Responsibility,Responsibility,Responsibility,Conflict,Conflict,Conflict,Conflict,Human Interest,Human Interest,...,Economic Consequences,Morality,Morality,Morality,Morality,WATER,WAR,PESTS/DISEASE,JOURNEY,BUILDINGS/STRUCTURE
0,0.914,0.978,0.001,0.581,0.991,0.994,0.995,0.039,0.016,0.208,...,0.040,0.003,0.982,0.091,0.963,0.984,0.994,0.984,0.990,0.985
1,0.990,0.988,0.978,0.224,0.981,0.997,0.978,0.002,0.978,0.779,...,0.008,0.855,0.956,0.976,0.941,0.984,0.991,0.984,0.987,0.963
2,0.012,0.041,0.006,0.053,0.071,0.076,0.916,0.157,0.070,0.346,...,0.005,0.933,0.970,0.263,0.823,0.030,0.339,0.064,0.034,0.113
3,0.088,0.967,0.004,0.010,0.022,0.132,0.330,0.263,0.005,0.004,...,0.002,0.135,0.076,0.015,0.375,0.035,0.096,0.093,0.041,0.029
4,0.003,0.381,0.000,0.027,0.142,0.400,0.873,0.216,0.034,0.008,...,0.003,0.003,0.085,0.005,0.855,0.878,0.969,0.980,0.886,0.226
5,0.003,0.005,0.001,0.016,0.002,0.002,0.003,0.001,0.014,0.180,...,0.005,0.002,0.011,0.009,0.002,0.027,0.074,0.022,0.010,0.019
6,0.013,0.070,0.002,0.030,0.005,0.002,0.005,0.002,0.003,0.062,...,0.004,0.005,0.013,0.005,0.009,0.030,0.132,0.023,0.009,0.035
7,0.001,0.003,0.000,0.003,0.000,0.000,0.001,0.001,0.001,0.858,...,0.001,0.000,0.002,0.001,0.001,0.007,0.016,0.011,0.003,0.004
8,0.004,0.001,0.001,0.002,0.001,0.000,0.002,0.000,0.004,0.318,...,0.000,0.001,0.001,0.002,0.001,0.021,0.051,0.024,0.010,0.025
9,0.074,0.985,0.001,0.497,0.986,0.996,0.989,0.989,0.005,0.567,...,0.027,0.004,0.921,0.016,0.599,0.979,0.994,0.992,0.983,0.972


In [29]:
# grouping based on frames
cols = list(set(scores.columns))
norm_scores = pd.DataFrame(columns=cols)
for c in cols:
    tmp = scores[c]
    if (len(tmp.shape) <= 1) :
         norm_scores[c]= tmp    
    else:
        number_frames = tmp.shape[1] 
        norm_scores[c]= tmp.sum(axis=1) / number_frames
norm_scores

,WAR,Economic Consequences,PESTS/DISEASE,WATER,Responsibility,JOURNEY,BUILDINGS/STRUCTURE,Human Interest,Conflict,Morality
0,0.994,0.23400,0.984,0.984,0.61850,0.990,0.985,0.29375,0.75475,0.50975
1,0.991,0.00675,0.984,0.984,0.79500,0.987,0.963,0.68050,0.73950,0.93200
2,0.339,0.00800,0.064,0.030,0.02800,0.034,0.113,0.19500,0.30500,0.74725
3,0.096,0.00250,0.093,0.035,0.26725,0.041,0.029,0.02925,0.18675,0.15025
4,0.969,0.00650,0.980,0.878,0.10275,0.886,0.226,0.07175,0.40775,0.23700
5,0.074,0.00350,0.022,0.027,0.00625,0.010,0.019,0.05100,0.00200,0.00600
6,0.132,0.00875,0.023,0.030,0.02875,0.009,0.035,0.01950,0.00350,0.00800
7,0.016,0.00075,0.011,0.007,0.00175,0.003,0.004,0.21600,0.00050,0.00100
8,0.051,0.00100,0.024,0.021,0.00200,0.010,0.025,0.08200,0.00075,0.00125
9,0.994,0.18725,0.992,0.979,0.38925,0.983,0.972,0.37950,0.99000,0.38500


## Hypothesis Tweaking

**Problem:** The original hypotheses gave very high entailment scores (0.96 to 0.99) for almost every item, including metaphor items that were not present in the text.

**Cause:** The hypotheses were long, abstract, and repeated "the refugee issue", so the model matched the topic instead of the specific claim. Controls ("football match", "cooking recipe") scored 0.0, so the model and code are fine.

**Potential Fix:** Rewrite each hypothesis as a short, concrete claim.